In [10]:
import uproot
import matplotlib.pyplot as plt
from scipy.interpolate import interp2d
import numpy as np
import healpy as hp
from tqdm import tqdm
import sys
import ROOT as rt
import root_numpy as rn

#root threeML astromodels numpy matplotlib healpy scipy astropy sympy uproot root_numpy 

In [21]:
data_name = "/data/home/cwy/Science/data.root"
file = uproot.open(data_name)
histogram_on = file["all_sky_cube_on;1"]
histogram_off = file["all_sky_cube_bg;1"]
header = file["header"].title.split(",")
header_value=file["header"].values()
print(header,header_value)

['live_time_w', 'live_time_k', 'zenith_max'] [   0.       1216.239092   50.      ]


In [25]:
nside=1024
npix=hp.nside2npix(nside)
dtype1 = [('count', float)]
dtype2 = [('count', float)]
skymaponout=np.zeros(npix) #, dtype = dtype1
skymapoffout=np.zeros(npix) #, dtype = dtype2
pixid = np.arange(npix)
pixarea= 4*np.pi/npix
new_lats = 90-hp.pix2ang(nside, pixid)[0]*180/np.pi # thetas I need to populate with interpolated theta values
new_lons = hp.pix2ang(nside, pixid)[1]*180/np.pi # phis, same

ras = histogram_on.axis(0).centers()
decs = histogram_on.axis(1).centers()

for bin in range(14):
    print('nHit%02d'%int(bin))
    skymap = histogram_on.values()[:,:,bin].T
    skymapoff = histogram_off.values()[:,:,bin].T
    fbkg = interp2d(ras,decs,skymapoff,kind='linear')
    
    for j in range(len(decs)):
        dec=decs[j]
        if (dec>=-25 and dec<=85):
            with open("/data/home/cwy/Science/3MLWCDA/Standard/src/tools/LHAASO_skytrans/skytxt/sky_bin%i_on%i.txt"%(bin, j),"a+") as fs:
                for i in tqdm(range(len(ras))):
                    ra=ras[i]
                    pick = (new_lons>ra-0.05) & (new_lons<ra+0.05) & (new_lats>dec-0.05) & (new_lats<dec+0.05)
                    pixneed = pixid[pick]
                    lens = np.sum(pick)
                    if lens:
                        poissonrand = np.random.poisson(skymap[i][j]/lens, lens)
                        for k in range(lens):
                            ra_pix , dec_pix = hp.pix2ang(1024,pixneed[k],lonlat=True)
                            coff = fbkg(ra_pix,dec_pix)/(np.radians(0.1)*np.radians(0.1)*np.cos(np.radians(dec_pix)))*pixarea
                            fs.write(str(0)+" "+str(pixneed[k])+" "+str(poissonrand[k])+"\n")
                            fs.write(str(1)+" "+str(pixneed[k])+" "+str(coff)+"\n")

nHit00


/tmp/ipykernel_30379/2541021225.py:19: DeprecationWarning: `interp2d` is deprecated!
`interp2d` is deprecated in SciPy 1.10 and will be removed in SciPy 1.13.0.

For legacy code, nearly bug-for-bug compatible replacements are
`RectBivariateSpline` on regular grids, and `bisplrep`/`bisplev` for
scattered 2D data.

In new code, for regular grids use `RegularGridInterpolator` instead.
For scattered data, prefer `LinearNDInterpolator` or
`CloughTocher2DInterpolator`.

For more details see
`https://scipy.github.io/devdocs/notebooks/interp_transition_guide.html`

  fbkg = interp2d(ras,decs,skymapoff,kind='linear')
  0%|          | 0/3600 [00:00<?, ?it/s]/tmp/ipykernel_30379/2541021225.py:34: DeprecationWarning:         `interp2d` is deprecated!
        `interp2d` is deprecated in SciPy 1.10 and will be removed in SciPy 1.13.0.

        For legacy code, nearly bug-for-bug compatible replacements are
        `RectBivariateSpline` on regular grids, and `bisplrep`/`bisplev` for
        scattered

KeyboardInterrupt: 

In [66]:
import taichi as ti
import numpy as np
import time
import sys

nside = 1024
npix = hp.nside2npix(nside)
skymaponout = np.zeros(npix)
pixid = np.arange(npix)
pixarea = 4 * np.pi / npix
new_lats = hp.pix2ang(nside, pixid)[0]
new_lons = hp.pix2ang(nside, pixid)[1]

ra = histogram_on.axis(0).centers()[:10]
dec = histogram_on.axis(1).centers()[:10]
skymap = histogram_on.values()[:, :, 0].T[:10,:10]

# Define data types for Taichi
dtype = ti.types.float32  # or ti.types.float64 based on your needs
dtype2 = ti.types.int32 
ti.init(arch=ti.cpu)

[Taichi] Starting on arch=x64


In [6]:
# maptree2 = "../../data/KM2a_testnew.root"
# file2 = uproot.open(maptree2)
# file2.keys()
# for i in range(14):
#     data = file2[f"nHit{i:02d}/data;1"].arrays(library="np")
#     bkg = file2[f"nHit{i:02d}/bkg;1"].arrays(library="np")
#     data = hp.ma(data["count"])
#     bkg = hp.ma(bkg["count"])
#     # hp.mollview(data)
#     # hp.mollview(bkg)
#     # hp.mollview(data-bkg)
#     skymap = histogram_on.values()[:,:,i].T
#     skymapoff = histogram_off.values()[:,:,i].T
#     print(i, np.ma.sum(data)/np.sum(skymap), np.ma.sum(bkg)/np.sum(skymapoff))

0 1.000000424584604 0.9999990138021146
1 0.9999989164353044 1.0000000253023262
2 0.9999997274414729 1.00000005253817
3 0.9999967806119096 1.0000005988871024
4 1.0000500597729831 1.0000014987974313
5 1.0000208190325064 1.0000068673662295
6 0.9998764148929981 1.0000437483845612
7 0.9999909770359822 1.0000288286949648
8 1.0007527940029188 1.0000169261050216
9 1.0000873760808782 1.000012650181002
10 0.9997341410732243 1.0000175463225096
11 1.001599859544866 1.0000164749461167
12 1.0017084607548472 1.0000333072978833
13 0.9977190295017662 1.0000292440544976


In [4]:
fout = rt.TFile.Open(f"./test.root", 'recreate')
tree = rt.TTree("BinInfo", "BinInfo")
name = bytearray(1)
duration = np.array([0], dtype=np.float64)
totalDuration = np.array([0], dtype=np.float64)
startMJD = np.array([0], dtype=np.float64)
stopMJD = np.array([0], dtype=np.float64)

# 将变量与branch关联
tree.Branch("name", name, "name/s")
tree.Branch("duration", duration, "duration/D")
tree.Branch("totalDuration", totalDuration, "totalDuration/D")
tree.Branch("startMJD", startMJD, "startMJD/D")
tree.Branch("stopMJD", stopMJD, "stopMJD/D")

# 填充数据
nside=1024
npix=hp.nside2npix(nside)
dtype1 = [('count', float)]
dtype2 = [('count', float)]
skymaponout=np.zeros(npix, dtype = dtype1)
skymapoffout=np.zeros(npix, dtype = dtype2)
pixid = np.arange(npix)
pixarea= 4*np.pi/npix


In [5]:
for i in range(14):
    name[0]=i
    duration[0] = 10
    totalDuration[0] = header_value[1]*24 #*0.99726956
    startMJD[0]=58844.0
    stopMJD[0]=60156.0
    tree.Fill()
fout.Write()
fout.Close()

In [6]:
for i in range(14):
    print('nHit%02d'%int(i))
    fout = rt.TFile.Open(f"~/Science/KM2A1234full_skymap.root", 'UPDATE')
    #skymap = file[f"hon_{i}"].values().T
    skymap = histogram_on.values()[:,:,i].T
    #plt.imshow(skymap, origin="lower", extent=[0,360,-20,80], aspect=True)
    skymapoff = histogram_off.values()[:,:,i].T
    ra = histogram_on.axis("x").centers()
    dec = histogram_on.axis("y").centers()
    RA, DEC=np.meshgrid(ra,dec)
    # plt.imshow(skymap, origin="lower", extent=[0,360,-90,90], aspect=True)
    fon = interp2d(ra,dec,skymap,kind='linear')
    fbkg = interp2d(ra,dec,skymapoff,kind='linear')
    
    ndir=fout.mkdir('nHit%02d'%int(i),'nHit%02d'%int(i))


    tree1=rt.TTree('data','data')
    tree2=rt.TTree('bkg','bkg')
    obj1 = rt.TParameter(int)("Nside",1024)
    obj2 = rt.TParameter(int)("Scheme",0)

    tree1.SetDirectory(ndir)
    tree1.SetEntries(npix)
    tree2.SetDirectory(ndir)
    tree2.SetEntries(npix)

    tree1.GetUserInfo().Add(obj1)
    tree1.GetUserInfo().Add(obj2)
    tree2.GetUserInfo().Add(obj1)
    tree2.GetUserInfo().Add(obj2)

    for ipix in tqdm(pixid):
        ra_pix , dec_pix = hp.pix2ang(1024,ipix,lonlat=True)
        if (dec_pix<-20.) | (dec_pix>80.):
            skymaponout[ipix]=hp.UNSEEN
            skymapoffout[ipix]=hp.UNSEEN
            continue
        skymaponout[ipix]=fon(ra_pix,dec_pix)/(np.radians(0.1)*np.radians(0.1)*np.cos(np.radians(dec_pix)))*pixarea
        skymapoffout[ipix]=fbkg(ra_pix,dec_pix)/(np.radians(0.1)*np.radians(0.1)*np.cos(np.radians(dec_pix)))*pixarea
    rn.array2tree(skymaponout,tree=tree1)
    rn.array2tree(skymapoffout,tree=tree2)
    fout.Write()
    fout.Close()

nHit00


/tmp/ipykernel_965/1847443822.py:12: DeprecationWarning: `interp2d` is deprecated!
`interp2d` is deprecated in SciPy 1.10 and will be removed in SciPy 1.12.0.

For legacy code, nearly bug-for-bug compatible replacements are
`RectBivariateSpline` on regular grids, and `bisplrep`/`bisplev` for
scattered 2D data.

In new code, for regular grids use `RegularGridInterpolator` instead.
For scattered data, prefer `LinearNDInterpolator` or
`CloughTocher2DInterpolator`.

For more details see
`https://gist.github.com/ev-br/8544371b40f414b7eaf3fe6217209bff`

  fon = interp2d(ra,dec,skymap,kind='linear')
/tmp/ipykernel_965/1847443822.py:13: DeprecationWarning: `interp2d` is deprecated!
`interp2d` is deprecated in SciPy 1.10 and will be removed in SciPy 1.12.0.

For legacy code, nearly bug-for-bug compatible replacements are
`RectBivariateSpline` on regular grids, and `bisplrep`/`bisplev` for
scattered 2D data.

In new code, for regular grids use `RegularGridInterpolator` instead.
For scattered dat

nHit01


100%|██████████| 12582912/12582912 [10:10<00:00, 20596.91it/s] 


nHit02


...

100%|██████████| 12582912/12582912 [09:50<00:00, 21318.64it/s] 


nHit12


100%|██████████| 12582912/12582912 [09:55<00:00, 21139.68it/s] 


nHit13


100%|██████████| 12582912/12582912 [10:27<00:00, 20038.85it/s] 
Error in <TList::Delete>: A list is accessing an object (0x55f8488b5270) already deleted (list name = UserInfo)
Error in <TList::Delete>: A list is accessing an object (0x55f8488ca730) already deleted (list name = UserInfo)
Error in <TList::Delete>: A list is accessing an object (0x55f848876710) already deleted (list name = UserInfo)
Error in <TList::Delete>: A list is accessing an object (0x55f8488c4530) already deleted (list name = UserInfo)
Error in <TList::Delete>: A list is accessing an object (0x55f845f174e0) already deleted (list name = UserInfo)
Error in <TList::Delete>: A list is accessing an object (0x55f842625c70) already deleted (list name = UserInfo)
Error in <TList::Delete>: A list is accessing an object (0x55f842625c70) already deleted (list name = UserInfo)
Error in <TList::Delete>: A list is accessing an object (0x55f8488ca730) already deleted (list name = UserInfo)
Error in <TList::Delete>: A list is acce

In [ ]:
import sys
sys.path.append("/data/home/rcy/data/km2a_data/3MLWCDA-master/Standard/src")
import mylib as my

In [ ]:
%matplotlib inline
map2, skymapHeader = hp.read_map("/data/home/rcy/data/km2a_data/mysig/KM2A_all_final_nHit000_0.69.fits.gz",h=True)
map2=hp.ma(map2)
my.hpDraw("region_name", "Modelname", map2,0,0,skyrange=(0,360,-20,80),
            colorlabel="Significance", contours=[10000000], save=False, cat={}, color="Milagro", xsize=2048)

In [ ]:
map2, skymapHeader = hp.read_map("/data/home/rcy/data/km2a_data/sg_data_nHit000_0.25.fits.gz",h=True, field=0)
map2=hp.ma(map2)
my.hpDraw("region_name", "Modelname", map2,0,0,skyrange=(0,360,-20,80),
            colorlabel="Significance", contours=[10000000], save=False, cat={}, color="Milagro", xsize=2048)

In [ ]:
29189.7/20479.6

In [ ]:
87.62*1.4253061583234048

In [3]:
import array
import ctypes
file_chnage = rt.TFile.Open(f"/data/home/cwy/Science/3MLWCDA/Standard/src/test.root", 'UPDATE')
tree = file_chnage.Get("BinInfo")

newtree = tree.CloneTree(0)
# name = array.array('d', [0])
# name = rt.TString()
# name = rt.std.string()
duration = array.array('d', [0])
totalDuration = array.array('d', [0])
# name = array.array('b', [0])
name = bytearray(1)
# 19103.1
newtree.SetBranchAddress("duration", duration)
newtree.SetBranchAddress("name", name)
newtree.SetBranchAddress("totalDuration", totalDuration)
# b = newtree.GetBranch("name")
# newtree.GetListOfBranches().Remove(b)
# newtree.Branch("name", name, 'name/C')  #rt.AddressOf(

for i in tqdm(range(14)):
    tree.GetEntry(i)
    newtree.GetEntry(i)
    print(i, name[0])
    duration[0] = 10
    totalDuration[0]=710.746617
    # name.assign(str(i))
    name[0]=i
    # name.SetString(str(i))
    newtree.Fill()
print("!")
file_chnage.Write("", rt.TObject.kOverwrite)
file_chnage.Close()

100%|██████████| 14/14 [00:00<00:00, 1346.70it/s]

0 0
1 0
2 1
3 2
4 3
5 4
6 5
7 6
8 7
9 8
10 9
11 10
12 11
13 12
!


In [ ]:
data_position="/data/home/rcy/data/km2a_data/data.root"
file = uproot.open(data_position)
file["header"].values()
# header_value_on[0]

In [ ]:
import array
import ctypes
file_chnage = rt.TFile.Open(f"/data/home/rcy/data/km2a_data/sg_data.root", 'UPDATE')
tree = file_chnage.Get("BinInfo")

newtree = tree.CloneTree(0)
# name = array.array('d', [0])
# name = rt.TString()
# name = rt.std.string()
duration = array.array('d', [0])
totalDuration = array.array('d', [0])
# name = array.array('b', [0])
name = bytearray(1)
newtree.SetBranchAddress("duration", duration)
newtree.SetBranchAddress("name", name)
newtree.SetBranchAddress("totalDuration", totalDuration)
# b = newtree.GetBranch("name")
# newtree.GetListOfBranches().Remove(b)
# newtree.Branch("name", name, 'name/C')  #rt.AddressOf(

for i in tqdm(range(14)):
    tree.GetEntry(i)
    newtree.GetEntry(i)
    print(i, name[0])
    duration[0] = 10
    totalDuration[0]=1216.239092*24
    # name.assign(str(i))
    name[0]=i
    # name.SetString(str(i))
    newtree.Fill()
print("!")
file_chnage.Write("", rt.TObject.kOverwrite)
file_chnage.Close()